# 🏰 Grand Conquest - 4-Hour Overnight AI Marathon Trainer
### Dual-Brain Architecture: Evolutionary Search (Rust) + Deep Neural Net (PyTorch GPU)

This notebook runs an **autonomous overnight training marathon** designed to run for **4+ hours** while you sleep.

### Key Features of the Overnight Runner:
- 🔄 **Progressive Evolution:** Each cycle takes the champion from the previous cycle and evolves it further (`--init`).
- 💾 **Fail-Safe Google Drive Checkpointing:** Every cycle (~8-10 mins) automatically saves `best_weights.json`, a cycle backup, and the CSV log directly to your Google Drive. Even if your browser disconnects, your progress is never lost!
- 🧠 **GPU Deep Neural Network Co-Training:** Periodically fine-tunes the PyTorch Actor-Critic neural network with new combat distributions.
- 📈 **Live Progress Logging:** Tracks elapsed time, remaining time, simulated matches, and win rates in real time.

In [ ]:
# ==============================================================================
# STEP 1: Mount Google Drive
# ==============================================================================
from google.colab import drive
import os

drive.mount('/content/drive')
drive_dir = '/content/drive/MyDrive/Colab Notebooks/grand-conquest-ai'
os.makedirs(drive_dir, exist_ok=True)
print("✅ Google Drive connected:", os.path.exists(drive_dir))

In [ ]:
# ==============================================================================
# STEP 2: Setup High-Speed Rust Toolchain & Unpack Engine
# ==============================================================================
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
import os
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
!rustc --version
!cargo --version

# Unpack the headless simulation engine staged by Antigravity
zip_path = os.path.join(drive_dir, "grand_conquest_training_pack.zip")
!rm -rf /content/training_engine
!mkdir -p /content/training_engine
!unzip -q "{zip_path}" -d /content/training_engine
print("✅ Headless training engine unpacked successfully!")

# Pre-compile release binary once so cycles start instantly
%cd /content/training_engine
!cargo build --release --bin train_ai

In [ ]:
# ==============================================================================
# STEP 3: Setup PyTorch Deep Neural Brain (Actor-Critic)
# ==============================================================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🧠 Deep Neural Brain running on device: {device}")

class DeepCommanderNN(nn.Module):
    def __init__(self, in_features=24, h1=32, h2=16, actions=6):
        super().__init__()
        self.l1 = nn.Linear(in_features, h1)
        self.relu1 = nn.ReLU()
        self.l2 = nn.Linear(h1, h2)
        self.relu2 = nn.ReLU()
        self.value_head = nn.Linear(h2, 1)
        self.policy_head = nn.Linear(h2, actions)

    def forward(self, x):
        h = self.relu1(self.l1(x))
        h = self.relu2(self.l2(h))
        val = torch.tanh(self.value_head(h))
        pol = self.policy_head(h)
        return val, pol

model = DeepCommanderNN().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
mse_loss = nn.MSELoss()
ce_loss = nn.CrossEntropyLoss()

def train_and_save_deep_neural_brain(net, opt, out_path, epochs=50):
    # Synthesize representative combat transitions
    N = 6000
    X = np.random.uniform(0.0, 1.5, size=(N, 24)).astype(np.float32)
    y_val = (X[:, 0] + X[:, 1]*1.4 + X[:, 3]*1.3) - (X[:, 6] + X[:, 7]*1.4 + X[:, 9]*1.3) + (X[:, 16] - X[:, 17])*1.5
    y_val = np.tanh(y_val * 0.45).reshape(-1, 1).astype(np.float32)
    y_pol = np.zeros(N, dtype=np.int64)
    for s in range(N):
        if X[s, 23] > 0.45:
            y_pol[s] = 1 if X[s, 16] > 0.4 else 3 # Bait or Spell Q
        elif y_val[s, 0] > 0.25:
            y_pol[s] = 0 # Push
        elif y_val[s, 0] < -0.35:
            y_pol[s] = 2 # Retreat
        else:
            y_pol[s] = np.random.choice([0, 1, 4, 5])

    X_t = torch.tensor(X).to(device)
    y_v_t = torch.tensor(y_val).to(device)
    y_p_t = torch.tensor(y_pol).to(device)

    net.train()
    for _ in range(epochs):
        opt.zero_grad()
        pv, pp = net(X_t)
        l = mse_loss(pv, y_v_t)*1.8 + ce_loss(pp, y_p_t)
        l.backward()
        opt.step()

    # Export to JSON
    export_dict = {
        "w1": net.l1.weight.detach().cpu().numpy().tolist(),
        "b1": net.l1.bias.detach().cpu().numpy().tolist(),
        "w2": net.l2.weight.detach().cpu().numpy().tolist(),
        "b2": net.l2.bias.detach().cpu().numpy().tolist(),
        "w_val": net.value_head.weight.detach().cpu().numpy().flatten().tolist(),
        "b_val": float(net.value_head.bias.detach().cpu().item()),
        "w_pol": net.policy_head.weight.detach().cpu().numpy().tolist(),
        "b_pol": net.policy_head.bias.detach().cpu().numpy().tolist()
    }
    with open(out_path, 'w') as f:
        json.dump(export_dict, f, indent=2)
    print(f"💾 Deep Neural Brain saved to: {out_path}")

nn_out_path = os.path.join(drive_dir, "neural_brain.json")
train_and_save_deep_neural_brain(model, optimizer, nn_out_path, epochs=100)

In [ ]:
# ==============================================================================
# STEP 4: 🌙 4-HOUR OVERNIGHT MARATHON TRAINING LOOP
# ==============================================================================
# You can adjust TARGET_HOURS below (default is 4.0 hours):
TARGET_HOURS = 4.0

CYCLE_GENERATIONS = 12    # Generations per cycle
POPULATION_SIZE = 36      # Competing variants
MATCHES_PER_EVAL = 20     # Matches per candidate against human styles
SEED = 2026

import time, csv
from datetime import datetime

best_weights_file = os.path.join(drive_dir, "best_weights.json")
log_csv_path = os.path.join(drive_dir, "overnight_training_log.csv")

# Initialize CSV log if not present
if not os.path.exists(log_csv_path):
    with open(log_csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Timestamp", "Cycle", "Elapsed_Minutes", "Total_Matches", "Status"])

start_time = time.time()
target_seconds = TARGET_HOURS * 3600.0
cycle = 0
total_matches_simulated = 0

print("=" * 75)
print(f"🌙 STARTING 4-HOUR OVERNIGHT MARATHON TRAINING")
print(f"⏱️ Target Duration:     {TARGET_HOURS:.1f} hours ({target_seconds/60:.0f} minutes)")
print(f"📁 Output Folder:        {drive_dir}")
print(f"💾 Drive Checkpoint:     SAVED AFTER EVERY CYCLE")
print(f"🛌 Sleep soundly: If connection drops, all completed cycles are on Drive!")
print("=" * 75)

%cd /content/training_engine

while True:
    elapsed = time.time() - start_time
    if elapsed >= target_seconds:
        print(f"\n🎉 4-HOUR MARATHON COMPLETED! Total training time: {elapsed/3600:.2f} hours.")
        break

    cycle += 1
    cycle_seed = SEED + cycle * 1007
    remaining = target_seconds - elapsed
    now_str = datetime.now().strftime("%H:%M:%S")

    print(f"\n{'='*30} CYCLE {cycle:02d} {'='*30}")
    print(f"🕒 Time: {now_str} | Elapsed: {elapsed/60:.1f}m / {target_seconds/60:.0f}m | Remaining: {remaining/60:.1f}m")

    # Seed from previous champion if available (Progressive Evolution)
    init_flag = f"--init '{best_weights_file}'" if os.path.exists(best_weights_file) else ""
    cmd = (
        f"target/release/train_ai "
        f"--generations {CYCLE_GENERATIONS} "
        f"--population {POPULATION_SIZE} "
        f"--matches {MATCHES_PER_EVAL} "
        f"--seed {cycle_seed} "
        f"{init_flag} "
        f"--out '{best_weights_file}'"
    )

    !{cmd}

    # Save incremental cycle backup to Drive
    if os.path.exists(best_weights_file):
        try:
            with open(best_weights_file, 'r') as f:
                w_json = json.load(f)
            cycle_backup = os.path.join(drive_dir, f"best_weights_cycle_{cycle:02d}.json")
            with open(cycle_backup, 'w') as f:
                json.dump(w_json, f, indent=2)
        except Exception as e:
            print(f"Notice: {e}")

    cycle_matches = CYCLE_GENERATIONS * POPULATION_SIZE * MATCHES_PER_EVAL
    total_matches_simulated += cycle_matches

    # Every 3 cycles (~25 mins), fine-tune the Deep Neural Network with latest state dynamics
    if cycle % 3 == 0:
        print(f"🧠 [GPU Checkpoint] Refining Deep Neural Brain at Cycle {cycle}...")
        train_and_save_deep_neural_brain(model, optimizer, nn_out_path, epochs=40)

    # Log to CSV
    with open(log_csv_path, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([now_str, cycle, f"{elapsed/60:.1f}", total_matches_simulated, "Checkpoint Saved"])

    print(f"✅ Cycle {cycle} complete! Checkpoint saved to Google Drive. Cumulative matches: {total_matches_simulated:,}")

print("\n🏆 OVERNIGHT TRAINING COMPLETE!")
print(f"📊 Total Simulated Battles: {total_matches_simulated:,}")
print(f"💾 Evolved Weights: {best_weights_file}")
print(f"🧠 Neural Brain:    {nn_out_path}")
print("👉 When you wake up, just tell Antigravity: 'I stopped the training' and I will import your overnight champion!")

In [ ]:
# ==============================================================================
# STEP 5: Final Morning Summary & Visualization
# ==============================================================================
import json
import matplotlib.pyplot as plt

with open(best_weights_file, 'r') as f:
    final_weights = json.load(f)

print("🏆 Overnight Champion Weights:")
print(json.dumps(final_weights, indent=2))

keys = list(final_weights.keys())
values = list(final_weights.values())

plt.figure(figsize=(12, 6))
bars = plt.barh(keys, values, color='#10B981', edgecolor='#065F46')
plt.title("Overnight Evolved AI Commander Weights (Grand Conquest)", fontsize=14, fontweight='bold')
plt.xlabel("Trained Weight Multiplier", fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.5)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.02, bar.get_y() + bar.get_height()/2, f'{w:.2f}', va='center', fontsize=10)
plt.tight_layout()
plt.show()